In [1]:
!git clone https://www.github.com/yousefkoriem/NN-Project1.git

Cloning into 'NN-Project1'...
remote: Enumerating objects: 190, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 190 (delta 28), reused 110 (delta 18), pack-reused 70 (from 1)
Receiving objects: 100% (190/190), 146.82 MiB | 16.95 MiB/s, done.
Resolving deltas: 100% (47/47), done.


In [2]:
import os
os.chdir("/content/NN-Project1")

In [3]:
import keras
import tensorflow as tf
import numpy as np
import pandas as pd
from keras import layers
from keras.utils import text_dataset_from_directory
import spacy
import re
import pickle
from keras import layers, models, metrics, optimizers,callbacks

In [4]:
model = models.load_model("models/architecture/untrained_imdb_model.keras")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adamw', because it has 44 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [5]:
checkpoint = callbacks.ModelCheckpoint(
    filepath="models/architecture/best_imdb_model.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [6]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

In [7]:
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [8]:
# 1. Load the raw datasets
train_ds = tf.data.Dataset.load("data/processed/train_ds")
val_ds = tf.data.Dataset.load("data/processed/val_ds")
test_ds = tf.data.Dataset.load("data/processed/test_ds")

# 2. Define the label shape fix
def vectorize_text(text, label):
    return text, tf.expand_dims(label, -1)

# 3. Map the fix to the data
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)

# 4. Apply cache and prefetch exactly ONCE at the very end
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [9]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

Epoch 1/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.5089 - f1_score: 0.6688 - loss: 0.7180
Epoch 1: val_loss improved from None to 0.69309, saving model to models/architecture/best_imdb_model.keras

Epoch 1: finished saving model to models/architecture/best_imdb_model.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 57s 79ms/step - accuracy: 0.5002 - f1_score: 0.6656 - loss: 0.7002 - val_accuracy: 0.5000 - val_f1_score: 0.6709 - val_loss: 0.6931 - learning_rate: 0.0010
Epoch 2/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.5036 - f1_score: 0.6688 - loss: 0.6934
Epoch 2: val_loss improved from 0.69309 to 0.69238, saving model to models/architecture/best_imdb_model.keras

Epoch 2: finished saving model to models/architecture/best_imdb_model.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 49s 78ms/step - accuracy: 0.5027 - f1_score: 0.6656 - loss: 0.6946 - val_accuracy: 0.5048 - val_f1_score: 0.6709 - val_loss: 0.6924 - learning_rate: 0.0010
Epoch 3/100
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 7

KeyboardInterrupt: 

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.to_csv("results/tables/history.csv", index=False)

In [ ]:
score = model.evaluate(test_ds)
score_df = pd.DataFrame([score], columns=model.metrics_names)

In [ ]:
model.save("results/models/final_imdb_model.keras")